In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()

In [0]:
def load_curated_agreement_profits(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "AgreementId string, SourceSystem string,"
        #     "PortfolioKey string, PortfolioName string,"
        #     "IngestionTimestamp timestamp, BatchId string, AgreementKey string,"
        #     "AgreementStatusCode string, AgreementStatus string, IssueDate date,"
        #     "ExpiredDate date, SignedDate date, Currency string,"
        #     "LimitAmount decimal(19,2), LimitUtilizedAmount decimal(19,2),"
        #     "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Agreement"

        # ================== Create Delta Table ==================

        # logging.info("Creating Delta Table...")
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "profits",
            )        
        else:
            logging.info("Tables doe not exist, no need to delete entries for business date!")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "unit_code_profits",
            "businessdate",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== Agreement ==================
        agreement_extra_filters = None

        if agreement_extra_filters is not None:
            agreement_extra_filters.append(date_filter)
            agreement_final_filters = " AND ".join(agreement_extra_filters)
        else:
            agreement_final_filters = date_filter

        agreement_selected_fields = [
            "agreement_number",
            "businessdate",
            "AGR_STATUS",
            "AGR_ISSUE_DT",
            "AGR_EXPIRY_DT",
            "AGR_SIGNING_DT",
            "CURRENCY",
            "AGR_LIMIT",
            "unit_code",
            "AGR_UTILISED_LIMIT",
        ]

        agreement = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_agreement",
            agreement_final_filters,
            agreement_selected_fields,
        )

        agreement_portfolio_join_condition = (
            agreement["unit_code"] == portfolio["unit_code_profits"]
        )

        # Joins
        target_table_profits = agreement.join(
            portfolio,
            agreement_portfolio_join_condition,
            "left",
        )

        agreement_primary_key_fields = [agreement.agreement_number, lit("profits")]

        target_table_profits = (
            target_table_profits.withColumn(
                "AgreementStatus",
                when(agreement.AGR_STATUS == 1, "Draft")
                .when(agreement.AGR_STATUS == 2, "Active")    #.when(agreement.AGR_STATUS == 1, "Active (Finalized)") #previous
                .when(agreement.AGR_STATUS == 3, "Denounced")
                .otherwise("Closed"),
            )
            .withColumn("SourceSystem", lit("profits"))
            .withColumn(
                "AgreementId", sha2(concat_ws(";", *agreement_primary_key_fields), 256)
            )
        )

        agreement_df = target_table_profits.select(
            target_table_profits.AgreementId,
            target_table_profits.SourceSystem,
            agreement.unit_code.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).cast("string").alias("BatchId"),
            agreement.agreement_number.alias("AgreementKey"),
            agreement.AGR_STATUS.alias("AgreementStatusCode"),
            target_table_profits.AgreementStatus.alias("AgreementStatus"),
            agreement.AGR_ISSUE_DT.alias("IssueDate"),
            agreement.AGR_EXPIRY_DT.alias("ExpiredDate"),
            agreement.AGR_SIGNING_DT.alias("SignedDate"),
            agreement.CURRENCY.alias("Currency"),
            agreement.AGR_LIMIT.cast("decimal(19,2)").alias("LimitAmount"),
            agreement.AGR_UTILISED_LIMIT.cast("decimal(19,2)").alias(
                "LimitUtilizedAmount"
            ),
            agreement.businessdate.alias("BusinessDate"),
        )

        # ================== Append ==================

        agreement_df.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_agreement_profits(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)